# UAW-AFS Frequency Points Contract

This notebook validates the planned job/worker contract for UAW-AFS frequency-point simulations without launching HFSS. It uses fake setup objects to show the two intended paths:

- no `frequency_points_ghz`: keep the template sweep, such as `LINC 3.5GHz 8.7GHz 401`;
- with `frequency_points_ghz`: replace the job sweep with a PyAEDT single-point sweep over exactly those GHz points.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path.cwd().parent
for import_root in (REPO_ROOT, REPO_ROOT / 'src'):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from worker.schemas import HfssSweepConfig, JobRequest
from worker.setup_designer import HfssSetupDesigner

REPO_ROOT


## Current Template Sweep

The existing template contract is still the default. For BCLF, the worker template typically defines a full interpolating sweep such as `LINC 3.5GHz 8.7GHz 401`.

In [11]:
template_sweep = HfssSweepConfig(
    sweep_type='interpolating',
    sweep_data='LINC 3.5GHz 8.7GHz 401',
)

job_default = JobRequest(
    job_id='uaw_afs_contract_default',
    template_name='bclf_template_v1',
    setup_name='Setup1',
    sweep_name='Sweep1',
    variables={'L': '7.8mm', 'L1': '8.1mm', 'L2': '7.9mm', 'W': '1.0mm'},
    output_filename='uaw_afs_contract_default.s2p',
    metrics_to_extract=['max_s11_db', 'min_s21_db'],
    submit_time='2026-06-01T00:00:00Z',
)

job_default.frequency_points_ghz is None, template_sweep

(True,
 HfssSweepConfig(sweep_type='interpolating', sweep_data='LINC 3.5GHz 8.7GHz 401', start_freq=None, stop_freq=None, point_count=None))

## Fake HFSS Objects

The fake objects record which PyAEDT-style method would be called by the worker. They do not solve or export anything.

In [12]:
class FakeSweep:
    def __init__(self, name):
        self.name = name
        self.props = {}


class FakeSetup:
    def __init__(self, name='Setup1'):
        self.name = name
        self.sweeps = []

    def get_sweep(self, name=None):
        if name is None:
            return self.sweeps[0] if self.sweeps else False
        return next((sweep for sweep in self.sweeps if sweep.name == name), False)

    def delete_sweep(self, name):
        for idx, sweep in enumerate(self.sweeps):
            if sweep.name == name:
                del self.sweeps[idx]
                return True
        return False

    def _add(self, name, kind):
        sweep = FakeSweep(name)
        sweep.props['kind'] = kind
        self.sweeps.append(sweep)
        return sweep

    def create_frequency_sweep(self, start_frequency=None, stop_frequency=None, num_of_freq_points=None, name=None, sweep_type='Discrete'):
        sweep = self._add(name or 'Sweep', 'frequency_sweep')
        sweep.props.update({
            'sweep_type': sweep_type,
            'start_frequency': start_frequency,
            'stop_frequency': stop_frequency,
            'num_of_freq_points': num_of_freq_points,
        })
        return sweep

    def create_linear_step_sweep(self, start_frequency=None, stop_frequency=None, step_size=None, name=None, sweep_type='Discrete'):
        sweep = self._add(name or 'Sweep', 'linear_step_sweep')
        sweep.props.update({
            'sweep_type': sweep_type,
            'start_frequency': start_frequency,
            'stop_frequency': stop_frequency,
            'step_size': step_size,
        })
        return sweep

    def create_single_point_sweep(self, name=None, unit='GHz', freq=None, save_single_field=False, save_fields=False):
        sweep = self._add(name or 'Sweep', 'single_point_sweep')
        sweep.props.update({
            'unit': unit,
            'freq': list(freq or []),
            'save_single_field': save_single_field,
            'save_fields': save_fields,
        })
        return sweep


class FakeHfss:
    def __init__(self):
        self.setups = []

    def create_setup(self, name):
        setup = FakeSetup(name)
        self.setups.append(setup)
        return setup

## Default Path: Keep Template Sweep

In [13]:
hfss = FakeHfss()
resolved = HfssSetupDesigner().ensure_setup(
    hfss,
    setup_name='Setup1',
    sweep_name='Sweep1',
    sweep_config=template_sweep,
)

sweep = hfss.setups[0].sweeps[0]
assert resolved.setup_name == 'Setup1'
assert sweep.props['kind'] == 'frequency_sweep'
assert sweep.props['start_frequency'] == '3.5GHz'
assert sweep.props['stop_frequency'] == '8.7GHz'
assert sweep.props['num_of_freq_points'] == 401
sweep.props

{'kind': 'frequency_sweep',
 'sweep_type': 'Interpolating',
 'start_frequency': '3.5GHz',
 'stop_frequency': '8.7GHz',
 'num_of_freq_points': 401}

## UAW-AFS Path: Override With Specific Frequency Points

`JobRequest` normalizes the optional GHz list by sorting and removing duplicates. The worker then creates a single-point sweep over those values.

In [14]:
job_uaw = JobRequest(
    job_id='uaw_afs_contract_points',
    template_name='bclf_template_v1',
    setup_name='Setup1',
    sweep_name='Sweep1',
    frequency_points_ghz=[5.4, 3.5, 4.8, 5.4, 8.7],
    variables={'L': '7.8mm', 'L1': '8.1mm', 'L2': '7.9mm', 'W': '1.0mm'},
    output_filename='uaw_afs_contract_points.s2p',
    metrics_to_extract=['max_s11_db', 'min_s21_db'],
    submit_time='2026-06-01T00:00:00Z',
)

hfss = FakeHfss()
HfssSetupDesigner().ensure_setup(
    hfss,
    setup_name='Setup1',
    sweep_name='Sweep1',
    sweep_config=template_sweep,
    frequency_points_ghz=job_uaw.frequency_points_ghz,
)

sweep = hfss.setups[0].sweeps[0]
assert job_uaw.frequency_points_ghz == [3.5, 4.8, 5.4, 8.7]
assert sweep.props['kind'] == 'single_point_sweep'
assert sweep.props['unit'] == 'GHz'
assert sweep.props['freq'] == [3.5, 4.8, 5.4, 8.7]
assert sweep.props['save_single_field'] is False
assert sweep.props['save_fields'] is False
sweep.props

{'kind': 'single_point_sweep',
 'unit': 'GHz',
 'freq': [3.5, 4.8, 5.4, 8.7],
 'save_single_field': False,
 'save_fields': False}

## Takeaway

The fake contract validates the intended data flow. A later real-HFSS smoke should submit one tiny `frequency_points_ghz` job and check that the exported Touchstone file contains exactly the requested frequency rows.

## Server-Side UAW-AFS Worker Job Submission

This section assumes the notebook runs on the server side. It selects one existing BCLF/UAW sample, draws 20 random frequency points from the same source frequency grid, and prepares a worker job with `frequency_points_ghz`. The notebook does not call PyAEDT directly; the Windows worker claims `queue/pending/<job_id>.json`, runs HFSS, uploads results, and the optional watcher can archive artifacts into `ext/afs`.

In [ ]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common.io_utils import read_json
from common.job_types import SimulationJob
from optimization import BCLFOptimizationSpec
from server.queue_manager import QueueManager
from src.poles_init import basic_poles_initialize
from src.utils import frequency_to_s
from src.vector_fitting import evaluate, vector_fit
from worker.schemas import ArtifactOptions, JobRequest

EXT_ROOT = REPO_ROOT / 'data' / 'bandpass_coupled_line_filter_export' / 'ext'
AFS_ROOT = EXT_ROOT / 'afs'
FALLBACK_RESULTS_ROOT = REPO_ROOT / 'queue' / 'results'
QUEUE_ROOT = REPO_ROOT / 'queue'
N_FREQ_POINTS = 20
RANDOM_SEED = 0
TEMPLATE_NAME = 'bclf_template_v1'
DESIGN_NAME = 'HFSSDesign1'
SETUP_NAME = 'Setup1'
SWEEP_NAME = 'Sweep1'
VARIABLE_ORDER = ['L', 'L1', 'L2', 'W', 'W1', 'W2', 'S1', 'S2']
BCLF_TOUCHSTONE_PREFIX = 'Bandpass_Coupled_Line_Filter_export_Bandpass_Coupled_Line_Filter'
EXPORT_CSV = False
RUN_SUBMIT_WORKER_JOB = True
WAIT_FOR_WORKER_RESULT = True
WORKER_TIMEOUT_SEC = 7200
WORKER_POLL_INTERVAL_SEC = 10

CONFIG_SUMMARY = {
    'ext_root': str(EXT_ROOT),
    'afs_root': str(AFS_ROOT),
    'fallback_results_root': str(FALLBACK_RESULTS_ROOT),
    'queue_root': str(QUEUE_ROOT),
    'n_freq_points': N_FREQ_POINTS,
    'random_seed': RANDOM_SEED,
    'job_id_policy': 'deterministic_source_seed_npoints',
    'template_name': TEMPLATE_NAME,
    'design_name': DESIGN_NAME,
    'setup_name': SETUP_NAME,
    'sweep_name': SWEEP_NAME,
    'run_submit_worker_job': RUN_SUBMIT_WORKER_JOB,
    'wait_for_worker_result': WAIT_FOR_WORKER_RESULT,
}
CONFIG_SUMMARY


In [16]:
def _is_touchstone(path: Path) -> bool:
    suffix = path.suffix.lower()
    return suffix.startswith('.s') and suffix.endswith('p')


def _decode_touchstone_value(encoded: str) -> float:
    text = encoded.strip()
    sign = -1.0 if text.startswith('m') else 1.0
    if text.startswith('m'):
        text = text[1:]
    return sign * float(text.replace('d', '.'))


def parse_bclf_variables_from_filename(path: Path, variable_order=VARIABLE_ORDER) -> dict[str, float]:
    parts = path.stem.split('_')
    parsed: dict[str, float] = {}
    expected = set(variable_order)
    idx = 0
    while idx < len(parts) - 1:
        key = parts[idx]
        if key in expected and key not in parsed:
            parsed[key] = _decode_touchstone_value(parts[idx + 1])
            idx += 2
        else:
            idx += 1
    missing = [name for name in variable_order if name not in parsed]
    if missing:
        raise ValueError(f'Could not parse variables {missing} from filename: {path.name}')
    return {name: float(parsed[name]) for name in variable_order}


def _variables_from_index(source_path: Path, ext_root: Path, variable_order=VARIABLE_ORDER) -> dict[str, float] | None:
    index_path = ext_root / 'index.json'
    if not index_path.exists():
        return None
    with index_path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    records = payload.get('records', {}) if isinstance(payload, dict) else {}
    if not isinstance(records, dict):
        return None
    for record in records.values():
        if not isinstance(record, dict):
            continue
        filename = record.get('filename')
        if filename is None or Path(str(filename)).name != source_path.name:
            continue
        variables = record.get('variables')
        if isinstance(variables, dict) and all(name in variables for name in variable_order):
            return {name: float(variables[name]) for name in variable_order}
    return None


def discover_source_touchstones(ext_root: Path, fallback_results_root: Path) -> tuple[list[Path], str]:
    if ext_root.exists():
        ext_candidates = sorted(
            path for path in ext_root.glob('*.s*p')
            if path.is_file() and _is_touchstone(path) and path.parent != (ext_root / 'afs')
        )
        if ext_candidates:
            return ext_candidates, 'ext'
    fallback_candidates = sorted(
        path for path in fallback_results_root.glob(f'{BCLF_TOUCHSTONE_PREFIX}*.s*p')
        if path.is_file() and _is_touchstone(path)
    )
    if fallback_candidates:
        return fallback_candidates, 'queue_results_fallback'
    raise FileNotFoundError(
        f'No source BCLF Touchstone files found in {ext_root} or {fallback_results_root}'
    )


def choose_source_sample(rng: np.random.Generator) -> tuple[Path, str, dict[str, float]]:
    candidates, source_kind = discover_source_touchstones(EXT_ROOT, FALLBACK_RESULTS_ROOT)
    selected = candidates[int(rng.integers(0, len(candidates)))]
    variables = _variables_from_index(selected, EXT_ROOT) if source_kind == 'ext' else None
    if variables is None:
        variables = parse_bclf_variables_from_filename(selected)
    return selected, source_kind, variables


def _touchstone_port_count(path: Path) -> int:
    suffix = path.suffix.lower()
    if not (suffix.startswith('.s') and suffix.endswith('p')):
        raise ValueError(f'Not a Touchstone file: {path}')
    return int(suffix[2:-1])


def _touchstone_complex(first: float, second: float, data_format: str) -> complex:
    if data_format == 'ri':
        return complex(first, second)
    if data_format == 'ma':
        return complex(first) * np.exp(1j * np.deg2rad(second))
    if data_format == 'db':
        return complex(10 ** (first / 20.0)) * np.exp(1j * np.deg2rad(second))
    raise ValueError(f'Unsupported Touchstone data format: {data_format}')


def read_touchstone_frequency_hz(path: Path) -> np.ndarray:
    freqs, _ = read_touchstone_network(path, include_s_params=False)
    return freqs


def read_touchstone_network(path: Path, *, include_s_params: bool = True) -> tuple[np.ndarray, np.ndarray | None]:
    # Small Touchstone v1 reader for .sNp files. This notebook only needs
    # frequency rows and optional S-parameter archival, so avoid requiring scikit-rf
    # in the server-side notebook kernel.
    unit_scale = {'hz': 1.0, 'khz': 1e3, 'mhz': 1e6, 'ghz': 1e9}
    unit = 'ghz'
    data_format = 'ma'
    n_ports = _touchstone_port_count(path)
    values_per_row = 1 + 2 * n_ports * n_ports
    freqs: list[float] = []
    rows: list[list[complex]] = []

    with path.open('r', encoding='utf-8', errors='ignore') as fh:
        for raw_line in fh:
            line = raw_line.split('!', 1)[0].strip()
            if not line:
                continue
            if line.startswith('#'):
                tokens = [token.lower() for token in line[1:].split()]
                for token in tokens:
                    if token in unit_scale:
                        unit = token
                    elif token in {'ri', 'ma', 'db'}:
                        data_format = token
                continue
            parts = line.split()
            if len(parts) < values_per_row:
                continue
            numbers = [float(part) for part in parts[:values_per_row]]
            freqs.append(numbers[0] * unit_scale[unit])
            if include_s_params:
                pairs = numbers[1:]
                rows.append([
                    _touchstone_complex(pairs[i], pairs[i + 1], data_format)
                    for i in range(0, len(pairs), 2)
                ])

    freq_array = np.asarray(freqs, dtype=float)
    if not include_s_params:
        return freq_array, None
    s_params = np.zeros((n_ports, n_ports, len(rows)), dtype=complex)
    for freq_idx, row in enumerate(rows):
        if n_ports == 2 and len(row) == 4:
            s_params[0, 0, freq_idx] = row[0]
            s_params[1, 0, freq_idx] = row[1]
            s_params[0, 1, freq_idx] = row[2]
            s_params[1, 1, freq_idx] = row[3]
        else:
            for flat_idx, value in enumerate(row):
                row_idx = flat_idx // n_ports
                col_idx = flat_idx % n_ports
                s_params[row_idx, col_idx, freq_idx] = value
    return freq_array, s_params


def sample_frequency_points_ghz(source_touchstone: Path, n_points: int, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    freq_hz = read_touchstone_frequency_hz(source_touchstone)
    if n_points > len(freq_hz):
        raise ValueError(f'n_points={n_points} exceeds available source frequencies={len(freq_hz)}')
    indices = np.sort(rng.choice(np.arange(len(freq_hz)), size=int(n_points), replace=False))
    sampled_hz = freq_hz[indices]
    sampled_ghz = sampled_hz / 1e9
    return sampled_hz, sampled_ghz


def make_afs_job_id(source_path: Path, seed: int, n_points: int) -> str:
    source_slug = source_path.stem[:48]
    source_hash = hashlib.sha1(source_path.name.encode('utf-8')).hexdigest()[:8]
    return f'uaw_afs_{source_slug}_{source_hash}_seed{int(seed)}_n{int(n_points)}'


def make_output_filename(job_id: str) -> str:
    return f'{job_id}.s2p'


def build_worker_job_payload(job_id: str, variables: dict[str, float], sampled_freq_ghz: np.ndarray) -> JobRequest:
    return JobRequest(
        job_id=job_id,
        template_name=TEMPLATE_NAME,
        design_name=DESIGN_NAME,
        setup_name=SETUP_NAME,
        sweep_name=SWEEP_NAME,
        frequency_points_ghz=[float(x) for x in sampled_freq_ghz],
        variables={name: f'{variables[name]:g}mm' for name in VARIABLE_ORDER},
        output_filename=make_output_filename(job_id),
        metrics_to_extract=['max_s11_db', 'min_s21_db'],
        artifact_options=ArtifactOptions(export_touchstone=True, export_csv=EXPORT_CSV),
        submit_time=datetime.now(timezone.utc),
        retry_count=0,
    )

In [17]:
rng = np.random.default_rng(RANDOM_SEED)
selected_touchstone, source_kind, selected_variables = choose_source_sample(rng)
sampled_freq_hz, sampled_freq_ghz = sample_frequency_points_ghz(selected_touchstone, N_FREQ_POINTS, rng)
source_freq_hz = read_touchstone_frequency_hz(selected_touchstone)
dry_run_job_id = make_afs_job_id(selected_touchstone, RANDOM_SEED, N_FREQ_POINTS)
dry_run_job = build_worker_job_payload(dry_run_job_id, selected_variables, sampled_freq_ghz)

DRY_RUN_SELECTION = {
    'source_kind': source_kind,
    'source_touchstone': str(selected_touchstone),
    'parsed_variables': selected_variables,
    'source_freq_count': int(len(source_freq_hz)),
    'source_freq_min_ghz': float(source_freq_hz[0] / 1e9),
    'source_freq_max_ghz': float(source_freq_hz[-1] / 1e9),
    'sampled_freq_count': int(len(sampled_freq_ghz)),
    'sampled_freq_ghz': [float(x) for x in sampled_freq_ghz],
    'worker_job_preview': dry_run_job.model_dump(mode='json'),
}

assert len(sampled_freq_ghz) == N_FREQ_POINTS
assert all(name in selected_variables for name in VARIABLE_ORDER)
assert dry_run_job.frequency_points_ghz == sorted(set(float(x) for x in sampled_freq_ghz))
DRY_RUN_SELECTION

{'source_kind': 'ext',
 'source_touchstone': '/home/zhanghf/projects/vector/data/bandpass_coupled_line_filter_export/ext/Bandpass_Coupled_Line_Filter_export_Bandpass_Coupled_Line_Filter_L_7d88_L1_8d0_L2_7d88_W_1d07_W1_0d9_W2_1d06_S1_0d14_S2_0d56.s2p',
 'parsed_variables': {'L': 7.88,
  'L1': 8.0,
  'L2': 7.88,
  'W': 1.07,
  'W1': 0.9,
  'W2': 1.06,
  'S1': 0.14,
  'S2': 0.56},
 'source_freq_count': 401,
 'source_freq_min_ghz': 3.5,
 'source_freq_max_ghz': 8.7,
 'sampled_freq_count': 20,
 'sampled_freq_ghz': [3.578,
  3.695,
  3.877,
  4.384,
  4.839,
  4.943,
  5.034,
  6.035,
  6.061,
  6.308,
  6.399,
  6.607,
  6.659,
  6.763,
  6.789,
  7.244,
  7.621,
  8.141,
  8.362,
  8.479],
 'worker_job_preview': {'job_id': 'uaw_afs_Bandpass_Coupled_Line_Filter_export_Bandpass_Cou_fcaa3c35_seed0_n20',
  'design_name': 'HFSSDesign1',
  'template_name': 'bclf_template_v1',
  'setup_name': 'Setup1',
  'sweep_name': 'Sweep1',
  'frequency_points_ghz': [3.578,
   3.695,
   3.877,
   4.384,
   4.8

## Submit To Worker Queue

Set `RUN_SUBMIT_WORKER_JOB = True` to write one canonical worker job JSON into `queue/pending`. If `WAIT_FOR_WORKER_RESULT` is also true, the notebook polls `queue/done` and `queue/failed`; when successful, it copies the worker Touchstone artifact and writes `.metadata.json` plus `.npz` under `ext/afs`.

In [18]:
def _simulation_job_from_request(job: JobRequest) -> SimulationJob:
    artifact_options = job.artifact_options.model_dump(mode='json')
    return SimulationJob(
        job_id=job.job_id,
        template_name=job.template_name,
        variables=dict(job.variables),
        output_filename=job.output_filename,
        metrics_to_extract=list(job.metrics_to_extract),
        submit_time=job.submit_time.isoformat(),
        design_name=job.design_name,
        setup_name=job.setup_name,
        sweep_name=job.sweep_name,
        frequency_points_ghz=list(job.frequency_points_ghz or []),
        artifact_options=artifact_options,
        retry_count=job.retry_count,
    )


def existing_worker_job_state(job_id: str) -> tuple[str | None, Path | None]:
    for state in ['pending', 'running']:
        path = QUEUE_ROOT / state / f'{job_id}.json'
        if path.exists():
            return state, path
    for state in ['done', 'failed']:
        path = QUEUE_ROOT / state / f'{job_id}.result.json'
        if path.exists():
            return state, path
    return None, None


def submit_worker_job(job: JobRequest, source_touchstone: Path, source_kind: str, variables: dict[str, float], sampled_hz: np.ndarray) -> dict:
    AFS_ROOT.mkdir(parents=True, exist_ok=True)
    queue = QueueManager(QUEUE_ROOT)
    for subdir in ['pending', 'running', 'done', 'failed', 'results']:
        (QUEUE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

    existing_state, existing_path = existing_worker_job_state(job.job_id)
    pending_path = None
    status = 'submitted'
    if existing_state is None:
        pending_path = queue.enqueue(_simulation_job_from_request(job))
    else:
        status = f'existing_{existing_state}'
        if existing_state == 'pending':
            pending_path = existing_path

    submission = {
        'status': status,
        'job_id': job.job_id,
        'pending_path': str(pending_path) if pending_path is not None else None,
        'existing_state': existing_state,
        'existing_path': str(existing_path) if existing_path is not None else None,
        'queue_root': str(QUEUE_ROOT),
        'source_kind': source_kind,
        'source_touchstone': str(source_touchstone),
        'variables': variables,
        'frequency_points_ghz': list(job.frequency_points_ghz or []),
        'frequency_points_hz': [float(x) for x in sampled_hz],
        'worker_job': job.model_dump(mode='json'),
        'submitted_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    submission_path = AFS_ROOT / f'{job.job_id}.submission.json'
    submission_path.write_text(json.dumps(submission, indent=2, ensure_ascii=True), encoding='utf-8')
    submission['submission_path'] = str(submission_path)
    return submission


def wait_for_worker_result(job_id: str, timeout_sec: int = WORKER_TIMEOUT_SEC, poll_interval_sec: int = WORKER_POLL_INTERVAL_SEC) -> tuple[Path, dict]:
    deadline = time.monotonic() + float(timeout_sec)
    done_path = QUEUE_ROOT / 'done' / f'{job_id}.result.json'
    failed_path = QUEUE_ROOT / 'failed' / f'{job_id}.result.json'
    while True:
        if done_path.exists():
            return done_path, read_json(done_path)
        if failed_path.exists():
            return failed_path, read_json(failed_path)
        if time.monotonic() >= deadline:
            raise TimeoutError(f'Timed out waiting for worker result: {job_id}')
        time.sleep(float(poll_interval_sec))


def archive_worker_result_to_afs(job: JobRequest, result_path: Path, result_payload: dict, submission: dict) -> dict:
    AFS_ROOT.mkdir(parents=True, exist_ok=True)
    artifacts = result_payload.get('artifacts') if isinstance(result_payload, dict) else None
    if not isinstance(artifacts, dict):
        artifacts = {}
    touchstone_rel = artifacts.get('touchstone')
    if not isinstance(touchstone_rel, str) or not touchstone_rel:
        raise ValueError(f'Worker result has no touchstone artifact: {result_path}')
    worker_touchstone = QUEUE_ROOT / touchstone_rel
    if not worker_touchstone.exists():
        raise FileNotFoundError(f'Worker touchstone artifact not found: {worker_touchstone}')

    afs_touchstone = AFS_ROOT / Path(job.output_filename).name
    shutil.copy2(worker_touchstone, afs_touchstone)
    csv_rel = artifacts.get('csv')
    afs_csv = None
    if isinstance(csv_rel, str) and csv_rel:
        worker_csv = QUEUE_ROOT / csv_rel
        if worker_csv.exists():
            afs_csv = AFS_ROOT / worker_csv.name
            shutil.copy2(worker_csv, afs_csv)

    resim_freq_hz, resim_s_params = read_touchstone_network(afs_touchstone)
    npz_path = AFS_ROOT / f'{job.job_id}.npz'
    np.savez_compressed(
        npz_path,
        freq_hz=resim_freq_hz,
        s_params=resim_s_params,
        variables=np.asarray([submission['variables'][name] for name in VARIABLE_ORDER], dtype=float),
        variable_order=np.asarray(VARIABLE_ORDER),
        source_touchstone=submission['source_touchstone'],
    )
    metadata = {
        **submission,
        'status': str(result_payload.get('status', result_path.parent.name)),
        'result_path': str(result_path),
        'result_payload': result_payload,
        'afs_touchstone': str(afs_touchstone),
        'afs_csv': str(afs_csv) if afs_csv is not None else None,
        'npz_path': str(npz_path),
        'archived_at_utc': datetime.now(timezone.utc).isoformat(),
        'resim_touchstone_freq_count': int(len(resim_freq_hz)),
    }
    metadata_path = AFS_ROOT / f'{job.job_id}.metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=True), encoding='utf-8')
    if metadata['resim_touchstone_freq_count'] != N_FREQ_POINTS:
        raise AssertionError(
            f"Expected {N_FREQ_POINTS} exported frequency rows, got {metadata['resim_touchstone_freq_count']}"
        )
    metadata['metadata_path'] = str(metadata_path)
    return metadata


def submit_uaw_afs_worker_job() -> dict:
    rng = np.random.default_rng(RANDOM_SEED)
    source_touchstone, source_kind, variables = choose_source_sample(rng)
    sampled_hz, sampled_ghz = sample_frequency_points_ghz(source_touchstone, N_FREQ_POINTS, rng)
    job_id = make_afs_job_id(source_touchstone, RANDOM_SEED, N_FREQ_POINTS)
    job = build_worker_job_payload(job_id, variables, sampled_ghz)
    submission = submit_worker_job(job, source_touchstone, source_kind, variables, sampled_hz)
    if not WAIT_FOR_WORKER_RESULT:
        return submission
    result_path, result_payload = wait_for_worker_result(job.job_id)
    if result_payload.get('status') != 'done':
        return {**submission, 'status': 'failed', 'result_path': str(result_path), 'result_payload': result_payload}
    return archive_worker_result_to_afs(job, result_path, result_payload, submission)


if RUN_SUBMIT_WORKER_JOB:
    WORKER_SUBMISSION_RESULT = submit_uaw_afs_worker_job()
else:
    WORKER_SUBMISSION_RESULT = {
        'status': 'skipped',
        'reason': 'Set RUN_SUBMIT_WORKER_JOB = True to write queue/pending job JSON for the worker.',
        'dry_run_source_touchstone': DRY_RUN_SELECTION['source_touchstone'],
        'dry_run_sampled_freq_count': DRY_RUN_SELECTION['sampled_freq_count'],
        'worker_job_preview': DRY_RUN_SELECTION['worker_job_preview'],
    }

WORKER_SUBMISSION_RESULT

{'status': 'done',
 'job_id': 'uaw_afs_Bandpass_Coupled_Line_Filter_export_Bandpass_Cou_fcaa3c35_seed0_n20',
 'pending_path': '/home/zhanghf/projects/vector/queue/pending/uaw_afs_Bandpass_Coupled_Line_Filter_export_Bandpass_Cou_fcaa3c35_seed0_n20.json',
 'existing_state': None,
 'existing_path': None,
 'queue_root': '/home/zhanghf/projects/vector/queue',
 'source_kind': 'ext',
 'source_touchstone': '/home/zhanghf/projects/vector/data/bandpass_coupled_line_filter_export/ext/Bandpass_Coupled_Line_Filter_export_Bandpass_Coupled_Line_Filter_L_7d88_L1_8d0_L2_7d88_W_1d07_W1_0d9_W2_1d06_S1_0d14_S2_0d56.s2p',
 'variables': {'L': 7.88,
  'L1': 8.0,
  'L2': 7.88,
  'W': 1.07,
  'W1': 0.9,
  'W2': 1.06,
  'S1': 0.14,
  'S2': 0.56},
 'frequency_points_ghz': [3.578,
  3.695,
  3.877,
  4.384,
  4.839,
  4.943,
  5.034,
  6.035,
  6.061,
  6.308,
  6.399,
  6.607,
  6.659,
  6.763,
  6.789,
  7.244,
  7.621,
  8.141,
  8.362,
  8.479],
 'frequency_points_hz': [3578000000.0,
  3695000000.0,
  3877000

## Compare Full-Wave Snp To Vector-Fitting Reconstruction

This section reuses the completed 20-point UAW-AFS worker result, fits a vector-fitting model for each S-parameter channel, reconstructs the response on the original full-wave frequency grid, and writes metrics, plots, and a reconstructed Touchstone file under `experiments/uaw_afs_single_case`.


In [ ]:
COMPARISON_OUTPUT_DIR = REPO_ROOT / 'experiments' / 'uaw_afs_single_case'
COMPARISON_PLOT_DIR = COMPARISON_OUTPUT_DIR / 'plots'
VF_REAL_POLES = 4
VF_COMPLEX_POLES = 8
VF_MAX_ITER = 30
VF_TOL = 1e-7
S_PARAM_CHANNELS = [
    ('S11', 0, 0),
    ('S21', 1, 0),
    ('S12', 0, 1),
    ('S22', 1, 1),
]
BCLF_COMPARISON_SPEC = BCLFOptimizationSpec(
    passband_ghz=(4.8, 5.4),
    lower_stopband_ghz=(3.6, 4.4),
    upper_stopband_ghz=(5.8, 8.7),
    min_passband_s21_db=-5.0,
    max_passband_s11_db=-7.0,
    max_stopband_s21_db=-15.0,
    s11_weight=0.5,
    stopband_weight=0.5,
)


def _resolve_existing_path(value) -> Path | None:
    if value is None:
        return None
    candidate = Path(str(value))
    if candidate.exists():
        return candidate
    if not candidate.is_absolute():
        repo_candidate = REPO_ROOT / candidate
        if repo_candidate.exists():
            return repo_candidate
    return None


def _read_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    if not isinstance(payload, dict):
        raise ValueError(f'Expected JSON object: {path}')
    return payload


def _candidate_afs_metadata_paths(job_id: str | None) -> list[Path]:
    paths: list[Path] = []
    if job_id:
        paths.append(AFS_ROOT / f'{job_id}.metadata.json')
    for path in sorted(AFS_ROOT.glob('uaw_afs_*_seed*_n*.metadata.json'), key=lambda item: item.stat().st_mtime, reverse=True):
        if path not in paths:
            paths.append(path)
    return paths


def _resolve_comparison_inputs() -> dict:
    worker_result = globals().get('WORKER_SUBMISSION_RESULT')
    if not isinstance(worker_result, dict):
        worker_result = {}

    job_id = worker_result.get('job_id')
    if not job_id:
        job_id = globals().get('dry_run_job_id')

    source_touchstone = _resolve_existing_path(
        worker_result.get('source_touchstone') or worker_result.get('dry_run_source_touchstone')
    )
    afs_touchstone = _resolve_existing_path(worker_result.get('afs_touchstone'))

    metadata_path = None
    metadata = None
    for candidate_path in _candidate_afs_metadata_paths(str(job_id) if job_id else None):
        candidate_metadata = _read_json_if_exists(candidate_path)
        if candidate_metadata is None:
            continue
        metadata_path = candidate_path
        metadata = candidate_metadata
        if not job_id:
            job_id = metadata.get('job_id') or candidate_path.stem.removesuffix('.metadata')
        source_touchstone = source_touchstone or _resolve_existing_path(metadata.get('source_touchstone'))
        afs_touchstone = afs_touchstone or _resolve_existing_path(metadata.get('afs_touchstone'))
        break

    if source_touchstone is None:
        dry_run_selection = globals().get('DRY_RUN_SELECTION')
        if isinstance(dry_run_selection, dict):
            source_touchstone = _resolve_existing_path(dry_run_selection.get('source_touchstone'))

    if afs_touchstone is None and metadata is not None:
        result_payload = metadata.get('result_payload')
        artifacts = result_payload.get('artifacts') if isinstance(result_payload, dict) else None
        if isinstance(artifacts, dict):
            afs_touchstone = _resolve_existing_path(artifacts.get('touchstone'))
            if afs_touchstone is None and isinstance(artifacts.get('touchstone'), str):
                afs_touchstone = _resolve_existing_path(QUEUE_ROOT / artifacts['touchstone'])

    if afs_touchstone is None and job_id:
        for candidate in [AFS_ROOT / f'{job_id}.s2p', QUEUE_ROOT / 'results' / f'{job_id}.s2p']:
            afs_touchstone = _resolve_existing_path(candidate)
            if afs_touchstone is not None:
                break

    if source_touchstone is None:
        raise FileNotFoundError('Could not resolve the original full-wave Touchstone path')
    if afs_touchstone is None:
        raise FileNotFoundError('Could not resolve the 20-point UAW-AFS Touchstone path')

    sampled_freq_ghz = None
    for payload in [metadata, worker_result, globals().get('DRY_RUN_SELECTION')]:
        if not isinstance(payload, dict):
            continue
        value = payload.get('frequency_points_ghz') or payload.get('sampled_freq_ghz')
        if value:
            sampled_freq_ghz = [float(item) for item in value]
            break

    return {
        'job_id': str(job_id) if job_id else afs_touchstone.stem,
        'source_touchstone': source_touchstone,
        'afs_touchstone': afs_touchstone,
        'metadata_path': metadata_path,
        'metadata': metadata or {},
        'sampled_freq_ghz': sampled_freq_ghz,
    }


def _sparam_db(values: np.ndarray) -> np.ndarray:
    return 20.0 * np.log10(np.maximum(np.abs(values), 1e-12))


def _fit_vector_reconstruction(sample_freq_hz: np.ndarray, sample_s_params: np.ndarray, full_freq_hz: np.ndarray) -> tuple[np.ndarray, pd.DataFrame]:
    reconstructed = np.zeros((sample_s_params.shape[0], sample_s_params.shape[1], len(full_freq_hz)), dtype=complex)
    fit_rows = []
    for channel, row, col in S_PARAM_CHANNELS:
        init_poles = basic_poles_initialize(
            freqs_range=(float(full_freq_hz[0]), float(full_freq_hz[-1])),
            n_poles_real=VF_REAL_POLES,
            n_poles_complex=VF_COMPLEX_POLES,
        )
        poles, residues, d, h = vector_fit(
            sample_freq_hz,
            sample_s_params[row, col, :],
            init_poles=init_poles,
            max_iter=VF_MAX_ITER,
            tol=VF_TOL,
            atol=VF_TOL,
            rtol=VF_TOL,
        )
        reconstructed[row, col, :] = evaluate(poles, residues, d, h, frequency_to_s(full_freq_hz))
        fit_rows.append({
            'channel': channel,
            'status': 'ok',
            'n_poles': int(len(poles)),
            'd': float(np.real(d)),
            'h': float(np.real(h)),
        })
    return reconstructed, pd.DataFrame(fit_rows)


def _metric_rows(full_s_params: np.ndarray, reconstructed_s_params: np.ndarray) -> list[dict]:
    rows = []
    for channel, row, col in S_PARAM_CHANNELS:
        true_values = full_s_params[row, col, :]
        pred_values = reconstructed_s_params[row, col, :]
        db_error = np.abs(_sparam_db(true_values) - _sparam_db(pred_values))
        complex_error = true_values - pred_values
        rows.append({
            'channel': channel,
            'db_mae': float(np.mean(db_error)),
            'db_rmse': float(np.sqrt(np.mean(db_error ** 2))),
            'db_max_error': float(np.max(db_error)),
            'complex_mae': float(np.mean(np.abs(complex_error))),
            'complex_rmse': float(np.sqrt(np.mean(np.abs(complex_error) ** 2))),
        })
    return rows


def _bclf_objective_value(freq_hz: np.ndarray, s_params: np.ndarray) -> float:
    return float(BCLF_COMPARISON_SPEC.evaluate(freq_hz, s_params).scalar_objective())


def _touchstone_row_values(s_params: np.ndarray, freq_idx: int) -> list[complex]:
    if s_params.shape[0] == 2 and s_params.shape[1] == 2:
        return [
            s_params[0, 0, freq_idx],
            s_params[1, 0, freq_idx],
            s_params[0, 1, freq_idx],
            s_params[1, 1, freq_idx],
        ]
    values = []
    for row in range(s_params.shape[0]):
        for col in range(s_params.shape[1]):
            values.append(s_params[row, col, freq_idx])
    return values


def write_touchstone_ri(path: Path, freq_hz: np.ndarray, s_params: np.ndarray) -> None:
    n_ports = int(s_params.shape[0])
    if s_params.shape[0] != s_params.shape[1]:
        raise ValueError(f'Touchstone export expects a square S-parameter matrix, got {s_params.shape}')
    lines = [
        '! Vector-fitting reconstruction from 20-point UAW-AFS data',
        '# Hz S RI R 50',
    ]
    for freq_idx, freq in enumerate(freq_hz):
        fields = [f'{float(freq):.12e}']
        for value in _touchstone_row_values(s_params, freq_idx):
            fields.append(f'{float(np.real(value)):.12e}')
            fields.append(f'{float(np.imag(value)):.12e}')
        lines.append(' '.join(fields))
    path.write_text('\n'.join(lines) + '\n', encoding='utf-8')


def _write_comparison_plots(job_id: str, full_freq_hz: np.ndarray, full_s_params: np.ndarray, sample_freq_hz: np.ndarray, sample_s_params: np.ndarray, reconstructed_s_params: np.ndarray) -> list[Path]:
    COMPARISON_PLOT_DIR.mkdir(parents=True, exist_ok=True)
    saved = []
    freq_ghz = full_freq_hz / 1e9
    sample_freq_ghz = sample_freq_hz / 1e9

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
    for ax, (channel, row, col) in zip(axes, S_PARAM_CHANNELS[:2]):
        ax.plot(freq_ghz, _sparam_db(full_s_params[row, col, :]), label=f'Original {channel}', linewidth=1.8)
        ax.plot(freq_ghz, _sparam_db(reconstructed_s_params[row, col, :]), label=f'VF reconstructed {channel}', linewidth=1.5, linestyle='--')
        ax.scatter(sample_freq_ghz, _sparam_db(sample_s_params[row, col, :]), s=18, label='20-point AFS samples', zorder=3)
        ax.set_ylabel(f'{channel} magnitude (dB)')
        ax.grid(alpha=0.25)
        ax.legend(loc='best')
    axes[-1].set_xlabel('Frequency (GHz)')
    response_path = COMPARISON_PLOT_DIR / f'{job_id}_s11_s21_original_vs_vf.png'
    fig.savefig(response_path, dpi=180)
    plt.close(fig)
    saved.append(response_path)

    fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
    for channel, row, col in S_PARAM_CHANNELS:
        db_error = np.abs(_sparam_db(full_s_params[row, col, :]) - _sparam_db(reconstructed_s_params[row, col, :]))
        ax.plot(freq_ghz, db_error, label=channel, linewidth=1.4)
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Absolute dB error')
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    error_path = COMPARISON_PLOT_DIR / f'{job_id}_all_channels_db_error.png'
    fig.savefig(error_path, dpi=180)
    plt.close(fig)
    saved.append(error_path)
    return saved


def run_uaw_afs_reconstruction_comparison() -> dict:
    COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    inputs = _resolve_comparison_inputs()
    full_freq_hz, full_s_params = read_touchstone_network(inputs['source_touchstone'])
    sample_freq_hz, sample_s_params = read_touchstone_network(inputs['afs_touchstone'])
    if full_s_params is None or sample_s_params is None:
        raise ValueError('Touchstone reader did not return S-parameter matrices')
    if len(sample_freq_hz) != N_FREQ_POINTS:
        raise AssertionError(f'Expected {N_FREQ_POINTS} sampled frequencies, got {len(sample_freq_hz)}')

    reconstructed_s_params, fit_df = _fit_vector_reconstruction(sample_freq_hz, sample_s_params, full_freq_hz)
    if reconstructed_s_params.shape != full_s_params.shape:
        raise AssertionError(f'Reconstructed shape {reconstructed_s_params.shape} does not match source shape {full_s_params.shape}')

    metrics_df = pd.DataFrame(_metric_rows(full_s_params, reconstructed_s_params))
    metric_cols = ['db_mae', 'db_rmse', 'db_max_error', 'complex_mae', 'complex_rmse']
    if not np.all(np.isfinite(metrics_df[metric_cols].to_numpy(dtype=float))):
        raise AssertionError('Non-finite reconstruction metrics detected')

    job_id = inputs['job_id']
    metrics_path = COMPARISON_OUTPUT_DIR / f'{job_id}_reconstruction_metrics.csv'
    fit_path = COMPARISON_OUTPUT_DIR / f'{job_id}_vf_fit_status.csv'
    reconstructed_touchstone_path = COMPARISON_OUTPUT_DIR / f'{job_id}_vf_reconstructed.s2p'
    summary_path = COMPARISON_OUTPUT_DIR / f'{job_id}_reconstruction_summary.json'

    metrics_df.to_csv(metrics_path, index=False)
    fit_df.to_csv(fit_path, index=False)
    write_touchstone_ri(reconstructed_touchstone_path, full_freq_hz, reconstructed_s_params)
    plot_paths = _write_comparison_plots(job_id, full_freq_hz, full_s_params, sample_freq_hz, sample_s_params, reconstructed_s_params)

    original_objective = _bclf_objective_value(full_freq_hz, full_s_params)
    reconstructed_objective = _bclf_objective_value(full_freq_hz, reconstructed_s_params)
    summary = {
        'job_id': job_id,
        'source_touchstone': str(inputs['source_touchstone']),
        'afs_touchstone': str(inputs['afs_touchstone']),
        'metadata_path': str(inputs['metadata_path']) if inputs['metadata_path'] is not None else None,
        'source_freq_count': int(len(full_freq_hz)),
        'sampled_freq_count': int(len(sample_freq_hz)),
        'reconstructed_freq_count': int(reconstructed_s_params.shape[2]),
        'sampled_freq_ghz': [float(x) for x in sample_freq_hz / 1e9],
        'metadata_sampled_freq_ghz': inputs['sampled_freq_ghz'],
        'vf_config': {
            'n_poles_real': VF_REAL_POLES,
            'n_poles_complex': VF_COMPLEX_POLES,
            'max_iter': VF_MAX_ITER,
            'tol': VF_TOL,
        },
        'objective': {
            'original': original_objective,
            'reconstructed': reconstructed_objective,
            'abs_error': float(abs(original_objective - reconstructed_objective)),
        },
        'metrics_csv': str(metrics_path),
        'fit_status_csv': str(fit_path),
        'reconstructed_touchstone': str(reconstructed_touchstone_path),
        'plots': [str(path) for path in plot_paths],
        'metrics': metrics_df.to_dict(orient='records'),
    }
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=True), encoding='utf-8')
    summary['summary_json'] = str(summary_path)
    return summary


UAW_AFS_RECONSTRUCTION_COMPARISON = run_uaw_afs_reconstruction_comparison()
UAW_AFS_RECONSTRUCTION_COMPARISON
